<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Loop de Entrenamiento</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">El loop de entrenamiento es la estructura repetitiva que aplica forward pass, cálculo de pérdida, retropropagación y actualización de parámetros una y otra vez. Este notebook se centra en esa mecánica.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Entender este bucle permite depurar, medir y adaptar el comportamiento real del aprendizaje.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A reconocer las etapas del loop de entrenamiento y cómo se encadenan dentro de una iteración estándar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Cuando el modelo no aprende, revisar el loop suele ser más útil que cambiar rápidamente la arquitectura.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Cada iteración del loop procesa un lote, calcula una pérdida y ajusta parámetros según los gradientes.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Tensores de entrada y etiquetas
"""Este bloque crea tensores que representan las entradas y las etiquetas para el entrenamiento.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "entradas" contiene muestras bidimensionales en forma de tensor que sirven como datos de entrada.

Aplicamos el método ".unsqueeze()" a las etiquetas para que tengan una dimensión adicional, convirtiéndolas en tensores columna.
El argumento "1" indica que se debe agregar una dimensión adicional en la posición 1 del tensor.
La variable "etiquetas" contiene valores asociados a cada muestra en forma de tensor con una dimensión adicional en 
la posición 1 y define el objetivo del entrenamiento.
Estos tensores permiten demostrar cómo se procesan los datos dentro del loop de entrenamiento."""

import torch

entradas = torch.tensor([[1.0, 2.0],
                         [3.0, 4.0],
                         [5.0, 6.0]])

etiquetas = torch.tensor([0.0, 1.0, 0.0]).unsqueeze(1)

entradas, etiquetas # Muestra los tensores de entrada y las etiquetas para verificar su correcta creación.


In [ ]:
# Bloque 2 — Definición de una red neuronal
"""Este bloque define una red neuronal artificial utilizando nn.Module.
Este bloque importa "torch.nn" para definir módulos y capas de redes neuronales.

Definimos una clase personalizada llamada "RedLoop()" que representa un modelo de red neuronal simple.
La clase "RedLoop()" hereda de "nn.Module", que es la clase base utilizada para construir redes neuronales en PyTorch.

Definimos la función "def __init__()", que inicializa la estructura interna del modelo.
El parámetro "self" de la función hace referencia a la instancia actual de la clase "RedLoop()".

Utilizamos "super().__init__()" para invocar el constructor de la clase base "nn.Module".     
Esto asegura que la clase base se inicialice correctamente cada vez que se crea una instancia del modelo.

Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa1" para definir la primera capa lineal.
Los argumentos "2, 4" de "nn.Linear()" indican que la capa lineal recibe 2 entradas y produce 4 salidas.
Aplicamos la clase "nn.Linear()" a la variable de instancia "self.capa2" para definir la segunda capa lineal.
Los argumentos "4, 1" de "nn.Linear()" indican que la capa lineal recibe 4 entradas y produce 1 salida.

Definimos la función "def forward()" que especifica cómo fluye la entrada a través de las capas del modelo.
El método ".forward()" especifica cómo se propaga la información a través de la red.
Los parámetros "self" y "x" representan la instancia del modelo y los datos de entrada, respectivamente.

Asignamos el resultado de llamar a "self.capa1()" a la variable "x", aplicando la primera capa lineal a la entrada.
Aplicamos la función de activación "torch.relu()" a la variable "x" para introducir no linealidad en la red.
Asignamos el resultado de llamar a "self.capa2()" a la variable "x", aplicando la segunda capa lineal a la salida de la activación.
Utilizamos la instrucción "return" para devolver la salida del modelo aplicando las capas lineales a la entrada.

En todos los casos se pasa "x" como parámetro a las funciones correspondientes para que la información fluya a través de la red.
Además, la variable "x" se actualiza con la salida de cada operación, asegurando que la información fluya correctamente a través de la red.
La variable "red" almacena una instancia de la clase "RedLoop()", representando el modelo de red neuronal definido.
La estructura permite demostrar cómo la red procesa datos durante el entrenamiento."""

import torch.nn as nn

class RedLoop(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(2, 4)
        self.capa2 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.capa1(x)
        x = torch.relu(x)
        x = self.capa2(x)
        return x

red = RedLoop()

red # Muestra la estructura del modelo.


In [ ]:
# Bloque 3 — Definición de la función de pérdida y el optimizador
"""Este bloque define la función de pérdida y el optimizador utilizados en el loop de entrenamiento.
Aplicamos la clase "nn.BCEWithLogitsLoss()" a la variable "criterio" para crear un objeto calculador de función de pérdida.
La variable "criterio" almacena el objeto calculador de la función de pérdida.

Aplicamos la clase "torch.optim.Adam()" a la variable "optimizador" para crear un objeto optimizador de parámetros de la red.
Pasamos como argumentos los parámetros de la red mediante la expresión "red.parameters()".
El método ".parameters()" devuelve un iterador sobre los parámetros de la red que serán actualizados por el optimizador.
El parámetro "lr=" especifica la tasa de aprendizaje utilizada por el optimizador, en este caso 0.01.
El optimizador Adam actualiza los parámetros de la red utilizando los gradientes calculados.
La variable "optimizador" almacena el objeto que gestiona las actualizaciones de los parámetros."""

criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.Adam(red.parameters(), lr=0.01)

criterio, optimizador # Muestra los objetos calculador de la función de pérdida y el optimizador.


In [ ]:
# Bloque 4 — Loop de entrenamiento
"""Este bloque implementa el loop de entrenamiento.
La variable "epocas" define cuántas veces se repite el ciclo completo, en este caso 5 veces.
La variable "perdidas" se inicializa como una lista vacía "[]" para almacenar los valores de pérdida de cada época en cada iteración del bucle.

Utilizamos un bucle "for" para iterar sobre el número de épocas definido en la variable "epocas", es decir, 5 veces en este caso.
La expresión "for _ in range(epocas):" inicia el bucle que se repetirá el número de veces definido en "epocas".

En cada iteración se ejecuta el forward pass, se calcula la pérdida, se reinician gradientes previos,
se ejecuta backward para obtener gradientes y se actualizan los parámetros con el optimizador.
Además, se almacena en la lista "perdidas" el valor calculado antes de actualizar los parámetros en cada iteración.
La variable "perdidas" almacena los valores de pérdida de cada época para su análisis posterior."""

epocas = 5
perdidas = []

for _ in range(epocas):
    predicciones = red(entradas)
    perdida = criterio(predicciones, etiquetas)
    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()
    perdidas.append(float(perdida.item()))


perdidas # Muestra la lista de pérdidas de cada época.


In [ ]:
# Bloque 5 — Resultados del loop de entrenamiento
"""Este bloque organiza los elementos principales del loop de entrenamiento.
La variable "resultado" almacena en forma de diccionario las pérdidas registradas y las predicciones del modelo después del entrenamiento.
La clave "predicciones_finales" contiene logits calculados con los parámetros finales y sin acumular gradientes.
La estructura permite visualizar cómo evoluciona el entrenamiento a lo largo de las iteraciones."""

red.eval()
with torch.no_grad():
    predicciones_finales = red(entradas)

resultado = {
    "perdidas_por_epoca": perdidas,
    "predicciones_finales": predicciones_finales.tolist()
}

resultado # Muestra el diccionario con las pérdidas por época y las predicciones de la última época.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto la estructura repetitiva que hace posible que la red aprenda a lo largo de muchas iteraciones.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">El loop traduce la teoría del entrenamiento en una secuencia operativa concreta.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Modificar partes del bucle sin comprobar cómo afecta al cálculo de pérdida, gradientes o actualización de pesos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en el Loop de Validación para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en el Loop de Validación y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">05_Entrenamiento_de_redes/03_El_loop_de_validacion.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">